# Lesson 10 - Agentes de IA em Produção

Nesta aula, você aprenderá **padrões de produção** para agentes de IA usando o Microsoft Agent Framework (Python). Abordamos:

- **Observabilidade** — adicionando temporização e registro às interações do agente
- **Avaliação** — usando um agente avaliador para pontuar a qualidade da resposta
- **Gerenciamento de custos** — estratégias para otimização de tokens e seleção de modelos

O cenário é um **agente de viagens** que ajuda os usuários a planejar viagens, com monitoramento e avaliação aplicados por cima.


## Configuração


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import time
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Azure AI Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Considerações sobre Produção

Levar agentes de IA de protótipos para produção requer atenção cuidadosa a três pilares:

1. **Observabilidade** — Você precisa de visibilidade sobre o que o agente está fazendo, quanto tempo leva e quais ferramentas ele utiliza. Sem rastreamento e registro de logs, depurar problemas em produção é quase impossível.

2. **Avaliação** — Verificações automatizadas de qualidade garantem que as respostas do agente permaneçam precisas, completas e úteis ao longo do tempo. Um agente avaliador pode pontuar as respostas com base em critérios definidos.

3. **Gerenciamento de Custos** — O uso de tokens impacta diretamente o custo. Estratégias como otimização de prompt, seleção de modelo e cache ajudam a manter as despesas sob controle sem sacrificar a qualidade.


## Construindo um Agente Observável

Definimos ferramentas de viagem e envolvemos a chamada do agente com uma medição de tempo para que possamos monitorar a latência. Em produção, você integraria com OpenTelemetry ou um backend de rastreamento similar.


In [ ]:
@tool(approval_mode="never_require")
def get_flight_info(destination: Annotated[str, "The destination city"]) -> str:
    """Get flight information for a destination."""
    flights = {
        "Paris": "BA 304, 08:30-11:45, $350",
        "Tokyo": "JL 044, 11:00-07:00+1, $890",
        "Barcelona": "VY 7821, 07:15-10:30, $280",
    }
    return flights.get(destination, f"No flights found to {destination}")


@tool(approval_mode="never_require")
def get_activity_suggestions(destination: Annotated[str, "The destination city"]) -> str:
    """Get activity suggestions for a destination."""
    activities = {
        "Paris": "Louvre Museum, Eiffel Tower, Seine River Cruise, Montmartre walking tour",
        "Tokyo": "Senso-ji Temple, Tsukiji Market tour, Shibuya Crossing, teamLab Borderless",
        "Barcelona": "Sagrada Familia, Park Güell, La Boqueria Market, Gothic Quarter walk",
    }
    return activities.get(destination, f"No activities found for {destination}")

In [ ]:
agent = client.as_agent(
    tools=[get_flight_info, get_activity_suggestions],
    name="TravelAgent",
    instructions="You are a helpful travel agent. Use the available tools to help users plan their trips. Provide comprehensive, actionable travel advice.",
)

# Simple observability: track timing
start_time = time.time()
response = await agent.run(
    "I want to plan a day trip in Paris. What flights and activities do you recommend?",
    )
elapsed = time.time() - start_time
print(f"Response ({elapsed:.2f}s):\n{response}")

## Padrões de Avaliação

Um padrão comum em produção é usar um segundo agente como **avaliador**. O avaliado atribui uma nota à resposta do agente principal com base em critérios pré-definidos, como completude, precisão e utilidade.

Isso possibilita:
- Portões automáticos de qualidade antes das respostas chegarem aos usuários
- Detecção de regressões quando os prompts ou modelos mudam
- Monitoramento contínuo do desempenho do agente ao longo do tempo


In [ ]:
evaluator = client.as_agent(
    name="ResponseEvaluator",
    instructions="""You evaluate travel agent responses on these criteria:
1. Completeness (1-5): Did it cover flights AND activities?
2. Accuracy (1-5): Is the information consistent?
3. Helpfulness (1-5): Would a traveler find this actionable?
4. Overall Score (1-5)
Provide scores and a brief explanation for each.""",
)

evaluation = await evaluator.run(f"Evaluate this travel agent response:\n\n{response}")
print(f"Evaluation:\n{evaluation}")

## Estratégias de Gerenciamento de Custos

Controlar os custos é fundamental para agentes de IA de produção. Aqui estão estratégias principais:

| Estratégia | Descrição |
|---|---|
| **Otimização de prompt** | Mantenha as instruções do sistema concisas. Remova contexto redundante para reduzir tokens de entrada. |
| **Seleção do modelo** | Use modelos menores e mais baratos (por exemplo, GPT-4o-mini) para tarefas simples como classificação ou extração, e reserve modelos maiores para raciocínio complexo. |
| **Cache** | Armazene os resultados das ferramentas e consultas frequentes em cache para evitar chamadas redundantes à API. |
| **Orçamentos de tokens** | Defina limites de `max_tokens` para evitar respostas inesperadamente longas. |
| **Agrupamento** | Agrupe várias consultas de usuários em uma única chamada de API quando for possível. |

Na prática, uma abordagem em níveis funciona bem: direcione solicitações simples para um modelo rápido e barato e escale apenas consultas complexas para um mais capaz.


## Summary

Nesta lição, você aprendeu a:

1. **Adicionar observabilidade** às interações do agente com temporização e registro, estabelecendo as bases para rastreamento e monitoramento.
2. **Avaliar respostas do agente** automaticamente usando um agente avaliador que pontua completude, precisão e utilidade.
3. **Gerenciar custos** por meio de otimização de prompts, seleção de modelo, cache e orçamentos de tokens.

Esses padrões de produção ajudam a garantir que seus agentes de IA sejam confiáveis, mensuráveis e econômicos em escala.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Aviso Legal**:
Este documento foi traduzido usando o serviço de tradução por IA [Co-op Translator](https://github.com/Azure/co-op-translator). Embora nos esforcemos pela precisão, por favor, esteja ciente de que traduções automatizadas podem conter erros ou imprecisões. O documento original em seu idioma nativo deve ser considerado a fonte autorizada. Para informações críticas, recomenda-se tradução profissional humana. Não nos responsabilizamos por quaisquer mal-entendidos ou interpretações incorretas decorrentes do uso desta tradução.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
